In [2]:
import warnings
warnings.filterwarnings('ignore')


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("ai_resume_screening.csv")

In [5]:
df.head()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
0,6,84.7,Bachelors,7,234,158,No
1,3,59.1,Masters,5,502,77,No
2,12,100.0,Masters,12,753,381,Yes
3,14,66.8,High School,8,529,407,Yes
4,10,99.6,Bachelors,10,754,331,Yes


In [6]:
df.tail()

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
29995,9,77.4,Bachelors,13,691,434,Yes
29996,5,77.8,Bachelors,9,473,149,No
29997,1,64.6,Bachelors,7,247,82,No
29998,7,94.8,Bachelors,12,584,409,Yes
29999,14,77.3,Masters,25,694,790,Yes


In [7]:
df

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted
0,6,84.7,Bachelors,7,234,158,No
1,3,59.1,Masters,5,502,77,No
2,12,100.0,Masters,12,753,381,Yes
3,14,66.8,High School,8,529,407,Yes
4,10,99.6,Bachelors,10,754,331,Yes
...,...,...,...,...,...,...,...
29995,9,77.4,Bachelors,13,691,434,Yes
29996,5,77.8,Bachelors,9,473,149,No
29997,1,64.6,Bachelors,7,247,82,No
29998,7,94.8,Bachelors,12,584,409,Yes


In [8]:
df.shape

(30000, 7)

In [9]:
df.columns

Index(['years_experience', 'skills_match_score', 'education_level',
       'project_count', 'resume_length', 'github_activity', 'shortlisted'],
      dtype='str')

In [10]:
df.info

<bound method DataFrame.info of        years_experience  skills_match_score education_level  project_count  \
0                     6                84.7       Bachelors              7   
1                     3                59.1         Masters              5   
2                    12               100.0         Masters             12   
3                    14                66.8     High School              8   
4                    10                99.6       Bachelors             10   
...                 ...                 ...             ...            ...   
29995                 9                77.4       Bachelors             13   
29996                 5                77.8       Bachelors              9   
29997                 1                64.6       Bachelors              7   
29998                 7                94.8       Bachelors             12   
29999                14                77.3         Masters             25   

       resume_length  github_ac

In [11]:
df.describe

<bound method NDFrame.describe of        years_experience  skills_match_score education_level  project_count  \
0                     6                84.7       Bachelors              7   
1                     3                59.1         Masters              5   
2                    12               100.0         Masters             12   
3                    14                66.8     High School              8   
4                    10                99.6       Bachelors             10   
...                 ...                 ...             ...            ...   
29995                 9                77.4       Bachelors             13   
29996                 5                77.8       Bachelors              9   
29997                 1                64.6       Bachelors              7   
29998                 7                94.8       Bachelors             12   
29999                14                77.3         Masters             25   

       resume_length  github_

In [12]:
df.dtypes

years_experience        int64
skills_match_score    float64
education_level           str
project_count           int64
resume_length           int64
github_activity         int64
shortlisted               str
dtype: object

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df[df.duplicated()]

,years_experience,skills_match_score,education_level,project_count,resume_length,github_activity,shortlisted


In [15]:
df = df[df['years_experience'] > 1]

In [16]:
edu_mapping = {'High School': 0, 'Bachelors': 1, 'Masters': 2, 'PhD': 3}
df['education_level_encoded'] = df['education_level'].map(edu_mapping)
df['shortlisted_label'] = df['shortlisted'].map({'Yes': 1, 'No': 0})

In [17]:
feature_cols = [
    'years_experience',
    'skills_match_score',
    'education_level_encoded',
    'project_count',
    'resume_length',
    'github_activity'
]

In [18]:
X = df[feature_cols]
y = df['shortlisted_label']

In [19]:
from sklearn.model_selection import train_test_split

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier



In [22]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

In [23]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [24]:
best_model = None
best_auc = 0.0
best_model_name = ""

print("\n--- Model Benchmark Results ---")
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    print(f"[{name}] Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f}")
    
    if auc > best_auc:
        best_auc = auc
        best_model = model
        best_model_name = name

print(f"\nSelected Best Model: {best_model_name} (ROC-AUC: {best_auc:.4f})")


--- Model Benchmark Results ---
[Logistic Regression] Accuracy: 0.9044 | F1: 0.9399 | ROC-AUC: 0.9572
[Random Forest] Accuracy: 0.8987 | F1: 0.9363 | ROC-AUC: 0.9526
[Gradient Boosting] Accuracy: 0.9067 | F1: 0.9413 | ROC-AUC: 0.9593

Selected Best Model: Gradient Boosting (ROC-AUC: 0.9593)


In [25]:
import joblib

In [26]:
artifact = {
    'model': best_model,
    'features': feature_cols,
    'edu_mapping': edu_mapping
}
joblib.dump(artifact, 'resume_screening_model.pkl')
print("Model saved to 'resume_screening_model.pkl'")

Model saved to 'resume_screening_model.pkl'
